<a href="https://colab.research.google.com/github/chuanbinp/uncertainty-aware-inference/blob/master/TeamB/teamb_vllm_profiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Team B — vLLM Throughput + PyTorch Profiler

## Mistral-7B PTQ Sweep

Runs **vLLM throughput benchmarking** and **PyTorch Profiler** analysis across all five quantization configs on the **same GPU instance**. Results are logged to W&B and saved for download.

| Config                 | Quantization     | vLLM Kernel                |
| ---------------------- | ---------------- | -------------------------- |
| `mistral-7b-fp16`      | FP16             | cuBLAS                     |
| `mistral-7b-gptq-int4` | GPTQ INT4        | gptq_marlin (+63% vs FP16) |
| `mistral-7b-gptq-int8` | GPTQ INT8        | gptq (exllama)             |
| `mistral-7b-awq-int4`  | AWQ INT4         | fused GEMV                 |
| `mistral-7b-nf4`       | NF4 bitsandbytes | HF batched fallback        |

Each benchmark is delegated to `run_vllm.py` or `run_profiler.py`  
(standalone scripts that live alongside this notebook in `TeamB/`).

**Datasets evaluated:** HellaSwag, TriviaQA, PubMedQA (handled in `run_eval.py` — not repeated here).  
**GPU requirement:** L4 (24 GB) or A100 (40 GB).


# 1. Setup


### 1.1 Verify GPU


In [ ]:
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {total_mem:.1f} GB")
    if total_mem < 20:
        print("WARNING: <20 GB VRAM — some configs may OOM. Recommend L4 or A100.")


Wed Apr 22 04:00:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             56W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### 1.2 Clone repo


In [ ]:
!git clone https://github.com/chuanbinp/uncertainty-aware-inference.git

fatal: destination path 'uncertainty-aware-inference' already exists and is not an empty directory.


### 1.3 Install core dependencies


In [ ]:
# Same package set as mistral_7b.ipynb — versions not changed
!pip install numpy plotly matplotlib torch transformers datasets accelerate gptqmodel netcal autoawq bitsandbytes


  Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
Using cached protobuf-7.34.1-cp310-abi3-manylinux2014_x86_64.whl (324 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.41.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
wandb 0.25.1 requires protobuf!=5.28.0,!=5.29.0,<7,>4.21.0, but you have protobuf 7.34.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 7.34.1 which is incompatible.
google-cloud-discovery

### 1.4 Install vLLM

vLLM is installed separately after core deps. `vllm==0.4.3` is compatible with the torch version above.


In [ ]:
# vllm==0.4.3 is outdated — Marlin INT4 autodetect unreliable before 0.5.x
!pip install "vllm>=0.6.0"


  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gptqmodel 6.0.3 requires protobuf>=7.34.0, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.29.0 requires opentelemetry-

### 1.5 Install W&B


In [ ]:
!pip install wandb

### 1.6 Verify scripts are present


In [ ]:
import os
REPO_DIR = "/content/uncertainty-aware-inference/TeamB"
for script in ["run_vllm.py", "run_profiler.py", "run_eval.py", "configs.py"]:
    path = os.path.join(REPO_DIR, script)
    status = "OK" if os.path.exists(path) else "MISSING — upload manually"
    print(f"  {script}: {status}")


  run_vllm.py: OK
  run_profiler.py: OK
  run_eval.py: OK
  configs.py: OK


# 2. Authentication


### 2.1 HuggingFace — required for gated Mistral-7B weights


In [ ]:
from huggingface_hub import login
from getpass import getpass
import os

HF_TOKEN = getpass("Enter your HuggingFace token: ")
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN   # propagate to subprocesses
print("HuggingFace login successful")


### 2.2 Weights & Biases


In [ ]:
import wandb
from getpass import getpass

WANDB_API_KEY = getpass("Enter your W&B API key (Enter to skip): ")
if WANDB_API_KEY.strip():
    wandb.login(key=WANDB_API_KEY)
    WANDB_ENABLED = True
    print("W&B login successful")
else:
    WANDB_ENABLED = False
    print("W&B disabled — results will be saved to JSON only")


# 3. Experiment Configuration


In [ ]:
import os
from pathlib import Path

REPO_DIR     = "/content/uncertainty-aware-inference/TeamB"
VLLM_OUT     = "/content/vllm_results"
PROFILER_OUT = "/content/profiler_results"

for d in [VLLM_OUT, PROFILER_OUT]:
    os.makedirs(d, exist_ok=True)

# All configs to sweep — same ordering as configs.py MODEL_REGISTRY
ALL_CONFIGS = [
    "mistral-7b-fp16",
    "mistral-7b-gptq-int4",
    "mistral-7b-gptq-int8",
    "mistral-7b-awq-int4",
    "mistral-7b-nf4",
]

WANDB_PROJECT = "UAI_Project"
WANDB_ENTITY  = "Uncertainty_Aware_Inference_Lab"

print(f"Sweep configs : {ALL_CONFIGS}")
print(f"vLLM output   : {VLLM_OUT}")
print(f"Profiler out  : {PROFILER_OUT}")
print(f"W&B enabled   : {WANDB_ENABLED}")


Sweep configs : ['mistral-7b-fp16', 'mistral-7b-gptq-int4', 'mistral-7b-gptq-int8', 'mistral-7b-awq-int4', 'mistral-7b-nf4']
vLLM output   : /content/vllm_results
Profiler out  : /content/profiler_results
W&B enabled   : True


# 4. Run Helpers


In [ ]:
import subprocess, sys, json, os
import torch
from pathlib import Path

# GPU spec auto-detection for roofline
GPU_SPECS = {
    "A100": {"tflops_fp16": 312.0,  "bandwidth_tb": 1.555},
    "L4":   {"tflops_fp16": 121.6,  "bandwidth_tb": 0.300},
    "T4":   {"tflops_fp16": 65.0,   "bandwidth_tb": 0.300},
    "V100": {"tflops_fp16": 125.0,  "bandwidth_tb": 0.900},
}

def detect_gpu_specs():
    if not torch.cuda.is_available():
        return {**GPU_SPECS["A100"], "name": "A100"}
    name = torch.cuda.get_device_name(0)
    for key, specs in GPU_SPECS.items():
        if key in name:
            print(f"Detected GPU: {name} -> using {key} specs")
            return {**specs, "name": key}
    print(f"WARNING: Unknown GPU {name!r}, defaulting to A100 specs.")
    return {**GPU_SPECS["A100"], "name": "unknown"}

GPU = detect_gpu_specs()
print(f"  TFLOPS FP16: {GPU['tflops_fp16']} | BW: {GPU['bandwidth_tb']} TB/s")

def run_script(script_path, extra_args):
    cmd = [sys.executable, script_path] + extra_args
    print(f"\nCMD: {chr(32).join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print("STDERR (tail):", result.stderr[-3000:])
    if result.returncode != 0:
        print(f"WARNING: script exited with code {result.returncode}")
    return result.returncode == 0

def load_json(path):
    p = Path(path)
    if not p.exists(): return None
    with open(p) as f: return json.load(f)

def wandb_run_init(config_key, experiment):
    if not WANDB_ENABLED: return None
    return wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name=f"teamB_{config_key}_{experiment}",
        reinit=True,
        config={"model": "mistral-7b", "team": "team-b",
                "config_key": config_key, "experiment": experiment},
    )

print("Helpers ready.")


Detected GPU: NVIDIA A100-SXM4-80GB -> using A100 specs
  TFLOPS FP16: 312.0 | BW: 1.555 TB/s
Helpers ready.


# 5. vLLM Throughput Benchmark

Calls `run_vllm.py` for each config. The script:

- Uses a fresh vLLM `LLM` instance per run (subprocess = clean GPU state)
- Runs 2 warmup passes then 3 timed passes over 10 benchmark prompts (128 output tokens each)
- NF4 automatically uses HF batched generation (vLLM does not support bitsandbytes)
- Saves `{config_key}_vllm.json` to `VLLM_OUT`

**Expected throughput on L4:**

- FP16 ≈ 1,090 tok/s (cuBLAS)
- GPTQ INT4 ≈ 1,780 tok/s (gptq_marlin — Marlin kernel auto-detected when `quantization=None`)
- GPTQ INT8 ≈ 640 tok/s (gptq exllama — no INT8 Marlin kernel)
- AWQ INT4 ≈ 1,020 tok/s (fused GEMV)
- NF4 ≈ 80–100 tok/s (HF batched fallback)


In [ ]:
import json
from pathlib import Path

SCRIPT_VLLM = os.path.join(REPO_DIR, "run_vllm.py")
vllm_summary = {}

for config_key in ALL_CONFIGS:
    print(f"\n" + "#"*65)
    print(f"# vLLM benchmark: {config_key}")
    print("#"*65)

    args = ["--config", config_key, "--output-dir", VLLM_OUT, "--hf-token", HF_TOKEN]
    # Do NOT pass --wandb-run-id: the subprocess would open a second
    # wandb.init() on the same run, causing duplicate metrics.
    # We log once below after reading the JSON.

    ok = run_script(SCRIPT_VLLM, args)
    if not ok:
        print(f"ERROR: run_vllm.py failed for {config_key}. Check STDERR above.")

    result = load_json(f"{VLLM_OUT}/{config_key}_vllm.json")
    if result:
        vllm_summary[config_key] = result
        print(f"DONE {config_key}: {result['avg_tok_per_sec']:.1f} tok/s | "
              f"kernel={result['kernel']} | mem={result['peak_gpu_mem_gb']:.2f} GB")
        if WANDB_ENABLED:
            run = wandb_run_init(config_key, "vllm_throughput")
            if run:
                run.log({"vllm/avg_tok_per_sec": result["avg_tok_per_sec"],
                         "vllm/peak_gpu_mem_gb": result["peak_gpu_mem_gb"],
                         "vllm/kernel":          result["kernel"]})
                run.finish()
    else:
        print(f"WARN: no JSON output for {config_key} — see stderr above")

print("\nvLLM benchmark complete.")


### 5.1 vLLM Results Summary


In [ ]:
import json
from pathlib import Path

COLS = ["Config", "Avg tok/s", "Peak GPU GB", "Kernel"]
rows = []
for cfg in ALL_CONFIGS:
    d = load_json(f"{VLLM_OUT}/{cfg}_vllm.json")
    if d:
        rows.append([cfg, f"{d['avg_tok_per_sec']:.2f}", f"{d['peak_gpu_mem_gb']:.3f}", d["kernel"]])
    else:
        rows.append([cfg, "N/A", "N/A", "N/A"])

w = [30, 12, 12, 35]
header = "  ".join(f"{c:<{w[i]}}" for i, c in enumerate(COLS))
print(header)
print("-" * sum(w))
for r in rows:
    print("  ".join(f"{v:<{w[i]}}" for i, v in enumerate(r)))


# 10. Download Results


In [ ]:
!zip -r /content/vllm_results.zip      /content/vllm_results/
print("Zipped vllm_results.zip and profiler_results.zip")


In [ ]:
from google.colab import files
import os

files.download("/content/vllm_results.zip")
files.download("/content/profiler_results.zip")
if os.path.exists("/content/roofline_mistral7b.png"):
    files.download("/content/roofline_mistral7b.png")
